#### Libraries, plot settings, utils

In [ ]:
# ================================
# Standard libraries
# ================================
import os
import re
import gc
from copy import deepcopy
from itertools import product

# ================================
# Numerical & scientific computing
# ================================
import numpy as np
import pandas as pd
import scipy
from scipy import signal, optimize, stats, integrate
from scipy.integrate import solve_ivp
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import acf

# ================================
# Machine learning / decomposition
# ================================
from sklearn.decomposition import PCA
import joblib
from joblib import Parallel, delayed

# ================================
# Optimization & root finding
# ================================
from scipy.optimize import fsolve

# ================================
# Plotting & visualization
# ================================
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns

# ================================
# Performance
# ================================
import numba as nb
from tqdm import tqdm

# ================================
# Correlation
# ================================
from numpy import correlate

# ================================
# Matplotlib plotting setup
# ================================
plt.style.use('tableau-colorblind10')  # colorblind-friendly palette
cmap = plt.get_cmap('Blues')           # colormap for plotting

# Default figure size and layout
plt.rcParams["figure.autolayout"] = True

# Font sizes
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "serif",
    "font.serif": "cmr10",
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True
})

# ================================
# Data and results folders
# ================================
folder_data = 'PLACEHOLDER/data/'
folder_results = 'PLACEHOLDER/results/'

#### Example plot of raw trace of activity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# ==========================
# Load data
# ==========================
data = np.load("/PLACEHOLDER/stimulus_traces_data.npz", allow_pickle=True)
time_axis = data["time_axis"]
full_calcium = data["full_calcium"]
full_spike = data["full_spike"]
stim_start = data["stim_start"]
stim_end = data["stim_end"]
stim_type = data["stim_type"]
stim_types_unique = data["stim_types_unique"]

# ==========================
# Set up figure
# ==========================
fig, ax = plt.subplots(figsize=(8/2.54, 6/2.54))

# Plot calcium trace
ax.plot(time_axis, full_calcium, color='black', alpha=1, label='Calcium')

# Optional: plot spike trace (commented out)
# ax.plot(time_axis, full_spike, color='k', alpha=0.7, label='Spike')

# ==========================
# Stimulus shading
# ==========================
palette = sns.color_palette("Set2", n_colors=len(stim_types_unique))
color_map = {stim: palette[i] for i, stim in enumerate(stim_types_unique)}

for start, end, typ in zip(stim_start, stim_end, stim_type):
    ax.axvspan(start, end, color=color_map[typ], alpha=0.25)

# ==========================
# Legend
# ==========================
patches = [mpatches.Patch(color=color_map[stim], label=stim) for stim in stim_types_unique]

ax.legend(
    handles=patches,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.15),
    ncol=len(stim_types_unique),
    frameon=True,
    columnspacing=0.8,
    handlelength=1.2,
    fontsize=9
)

# ==========================
# Axis styling
# ==========================
ax.set_xticks([5000, 5200, 5400])
ax.set_yticks([])
ax.set_ylabel('Activity $\Delta F/F_0$')
ax.set_xlabel('Time [s]')
ax.set_ylim(-100, 350)
ax.set_xlim(4960, 5400)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

# ==========================
# Show or save
# ==========================
# plt.savefig('/home/zenari/RNN_heterogeneity/results/example_traces.png',
#             dpi=1000, bbox_inches='tight')
plt.show()

#### Experimental acfs

In [ ]:
import numpy as np
import pandas as pd
from numpy import correlate

# -----------------------------
# Load data
# -----------------------------
df_all = pd.read_pickle("/data/zenari/RNN_heterogeneity/Dati_Notebook_RNN/final_df.pkl")
df_matched = pd.read_pickle("/home/zenari/hwhm_in_degree_df_proof_1412.pkl")
df_degrees = pd.read_pickle("/home/zenari/degree_df_proof_1412.pkl")

# -----------------------------
# Functions
# -----------------------------
def autocorr(x, maxlag):
    """
    Compute the autocorrelation of a 1D array `x` up to `maxlag`.
    """
    x0 = x - np.mean(x)
    corr = correlate(x0, x0, mode='full')
    mid = len(corr) // 2
    return corr[mid:mid+maxlag+1] / corr[mid]

def compute_hwhm(ac):
    """
    Compute the half-width at half-maximum (HWHM) of an autocorrelation function `ac`.
    Returns the lag (index) where ACF first drops below 0.5.
    """
    half_max = 0.5
    below_half = np.where(ac < half_max)[0]
    if len(below_half) == 0:
        return np.nan
    return below_half[0]

def compute_trace_hwhm(row, trace_type="calcium", maxlag=250):
    """
    Compute HWHM from a trace (calcium or rate). Returns HWHM in seconds.
    """
    trace = row['calcium_trace'] if trace_type=="calcium" else row['rate_trace']
    fps = row['fps']
    
    if trace is None or len(trace) < maxlag:
        return np.nan

    ac = autocorr(trace, maxlag=maxlag)
    hwhm_samples = compute_hwhm(ac)
    return hwhm_samples / fps if not np.isnan(hwhm_samples) else np.nan

# -----------------------------
# Compute ACFs for all calcium traces
# -----------------------------
df_all['acf'] = df_all.apply(
    lambda row: autocorr(row['calcium_trace'], maxlag=250) 
    if row['calcium_trace'] is not None and len(row['calcium_trace']) >= 250 else np.nan, 
    axis=1
)

# -----------------------------
# Bin by in-degree
# -----------------------------
bins = np.percentile(df_all['in_degree'], [0, 20, 40, 60, 80, 100])
df_all['bin'] = pd.cut(df_all['in_degree'], bins=bins, include_lowest=True)

# -----------------------------
# Compute mean ACF per bin
# -----------------------------
mean_acfs = (
    df_all.groupby('bin')['acf']
    .apply(lambda x: np.vstack(x.dropna().to_list()).mean(axis=0))
)

#### Structural data analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

# -----------------------------
# Truncated lognormal negative log-likelihood
# -----------------------------
def truncated_lognormal_nll(params, data, kmin):
    """Negative log-likelihood for a truncated lognormal."""
    mu, sigma = params
    if sigma <= 0:
        return np.inf

    x = np.log(data)
    zmin = (np.log(kmin) - mu) / sigma

    logpdf = stats.norm.logpdf(x, loc=mu, scale=sigma) - np.log(data)
    logZ = np.log(1 - stats.norm.cdf(zmin))

    return -(np.sum(logpdf) - len(data) * logZ)

# -----------------------------
# Fit truncated lognormal
# -----------------------------
def fit_truncated_lognormal(data, kmin):
    data = data[data >= kmin]
    if len(data) < 20:
        return None

    mu0 = np.mean(np.log(data))
    sigma0 = np.std(np.log(data))

    res = optimize.minimize(
        truncated_lognormal_nll,
        x0=(mu0, sigma0),
        args=(data, kmin),
        bounds=[(None, None), (1e-6, None)]
    )

    if not res.success:
        return None
    return res.x  # mu, sigma

# -----------------------------
# KS distance for truncated lognormal
# -----------------------------
def ks_distance_truncated_lognormal(data, mu, sigma, kmin):
    data = np.sort(data[data >= kmin])
    n = len(data)

    z = (np.log(data) - mu) / sigma
    zmin = (np.log(kmin) - mu) / sigma

    model_cdf = (stats.norm.cdf(z) - stats.norm.cdf(zmin)) / (1 - stats.norm.cdf(zmin))
    empirical_cdf = np.arange(1, n + 1) / n

    return np.max(np.abs(empirical_cdf - model_cdf))

# -----------------------------
# Evaluate multiple kmin cutoffs
# -----------------------------
def estimate_kmin_truncated_lognormal(data, kmin_values):
    results = []
    for kmin in kmin_values:
        fit = fit_truncated_lognormal(data, kmin)
        if fit is None:
            continue
        mu, sigma = fit
        ks = ks_distance_truncated_lognormal(data, mu, sigma, kmin)
        results.append({
            "kmin": kmin, "mu": mu, "sigma": sigma, "ks": ks,
            "n": np.sum(data >= kmin)
        })
    return results

# -----------------------------
# Truncated lognormal PDF
# -----------------------------
def truncated_lognormal_pdf(x, mu, sigma, kmin):
    z = (np.log(x) - mu) / sigma
    zmin = (np.log(kmin) - mu) / sigma
    pdf = stats.norm.pdf(z) / (x * sigma)
    Z = 1 - stats.norm.cdf(zmin)
    return pdf / Z

# -----------------------------
# Candidate kmin cutoffs
# -----------------------------
kmin_values = np.unique(np.percentile(data, np.linspace(5, 40, 30)).astype(int))
results = estimate_kmin_truncated_lognormal(data, kmin_values)
best = min(results, key=lambda r: r["ks"])

mu, sigma, kmin = best["mu"], best["sigma"], best["kmin"]

print(f"Best kmin: {kmin}, mu: {mu:.3f}, sigma: {sigma:.3f}, KS distance: {best['ks']:.3f}, N used: {best['n']}")

# -----------------------------
# KS distance plot
# -----------------------------
plt.figure(figsize=(8/2.54, 6/2.54))
plt.plot([r["kmin"] for r in results], [r["ks"] for r in results], marker="o")
plt.xlabel(r"$k_{\min}$")
plt.ylabel("KS distance")
plt.tight_layout()
plt.savefig('/PLACEHOLDER/ks_distance_truncated_lognormal.png', dpi=300)

# -----------------------------
# Histogram with truncated lognormal fit
# -----------------------------
plt.figure(figsize=(8/2.54, 6/2.54))
bins = np.logspace(np.log10(max(1, data.min())), np.log10(data.max()), 50)

plt.hist(data[data < kmin], bins=bins, density=True, alpha=0.35, color="tab:gray", label=r"$k < k_{\min}$")
plt.hist(data[data >= kmin], bins=bins, density=True, alpha=0.6, color="tab:blue", label=r"$k \geq k_{\min}$")

x_fit = np.logspace(np.log10(kmin), np.log10(data.max()), 500)
pdf_fit = truncated_lognormal_pdf(x_fit, mu, sigma, kmin)
plt.plot(x_fit, pdf_fit, color="black", linewidth=2)
plt.axvline(kmin, linestyle="--", color="black", linewidth=1)

plt.xscale("log")
plt.yscale("log")
plt.ylim(1e-5, 1)
plt.xlabel(r"$k$ (in-degree)")
plt.ylabel("Probability density")
plt.legend(loc='upper right', frameon=False)
plt.tight_layout()
plt.savefig('/PLACEHOLDER/truncated_lognormal_fit.png', dpi=300)

# -----------------------------
# Empirical CCDF vs fit
# -----------------------------
def ccdf(x):
    x = np.sort(x)
    return x, 1 - np.arange(1, len(x)+1)/len(x)

plt.figure(figsize=(8/2.54, 6/2.54))
kept_data = data[data >= kmin]
x_emp, y_emp = ccdf(kept_data)
plt.loglog(x_emp, y_emp, '.', label="Empirical")

z = (np.log(x_fit) - mu) / sigma
zmin = (np.log(kmin) - mu) / sigma
model_ccdf = 1 - (stats.norm.cdf(z) - stats.norm.cdf(zmin)) / (1 - stats.norm.cdf(zmin))
plt.loglog(x_fit, model_ccdf, linewidth=2, label="Fit")

plt.xlabel(r"$k$ (in-degree)")
plt.ylabel("CCDF")
plt.legend()
plt.tight_layout()
plt.savefig('/PLACEHOLDER/truncated_lognormal_ccdf.png', dpi=300)

#### Model data generation

In [ ]:
# ============================================================
# PyTorch CUDA lognormal RNN simulations
# ============================================================

import torch
import numpy as np
from statsmodels.tsa.stattools import acf

# -----------------------------
# Device
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float32)

# -----------------------------
# Activation function
# -----------------------------
def phi(x):
    return torch.tanh(x)

# -----------------------------
# Sample lognormal degrees
# -----------------------------
def sample_lognormal(mu, sigma, N):
    return np.random.lognormal(mean=mu, sigma=sigma, size=N)

# -----------------------------
# Interaction matrix
# -----------------------------
def inter_matrix(K, N, g, gamma, device):
    """Generate a correlated interaction matrix for RNN."""
    z = torch.randn(N, N, 2, device=device)
    J = torch.zeros((N, N), device=device)
    
    iu = torch.triu_indices(N, N, offset=1, device=device)
    i, j = iu[0], iu[1]
    
    scale = g / torch.sqrt(torch.tensor(float(K), device=device))
    gamma_perp = torch.sqrt(1.0 - gamma**2)
    
    J[i, j] = scale * z[i, j, 0]
    J[j, i] = scale * (gamma * z[i, j, 0] + gamma_perp * z[i, j, 1])
    
    return J

# -----------------------------
# Adjacency matrix
# -----------------------------
def adjacency_matrix(N, K, degree_sequence, device):
    deg = torch.tensor(degree_sequence, device=device)
    prob = torch.outer(deg, deg) / (K * N)
    
    A = torch.zeros((N, N), device=device)
    iu = torch.triu_indices(N, N, offset=1, device=device)
    
    mask = torch.rand(len(iu[0]), device=device) < prob[iu[0], iu[1]]
    A[iu[0][mask], iu[1][mask]] = 1
    A = A + A.T
    return A

# -----------------------------
# Euler step
# -----------------------------
def step_dynamics(x, W, dt):
    return x + dt * (-x + torch.matmul(W, phi(x)))

# -----------------------------
# HWHM of autocorrelation
# -----------------------------
def HWHM(acf_values):
    acf_values = np.asarray(acf_values)
    if len(acf_values) < 2 or not np.isfinite(acf_values).all():
        return -1.0
    
    half = acf_values[0] / 2
    idx = np.where(acf_values < half)[0]
    
    if len(idx) == 0 or idx[0] == 0:
        return -1.0
    
    i = idx[0]
    x0, x1 = i - 1, i
    y0, y1 = acf_values[x0], acf_values[x1]
    return x0 + (half - y0) / (y1 - y0)

# -----------------------------
# Autocorrelation simulation
# -----------------------------
def ac_sim(traj, dt, nlags):
    T, N = traj.shape
    taus = np.zeros(N)
    for i in range(N):
        C = acf(traj[:, i].numpy(), nlags=nlags - 1)
        taus[i] = HWHM(C) * dt
    return taus

# -----------------------------
# Main simulation function
# -----------------------------
def simulate_lognormal_torch(params, hyperparams):
    N, g, gamma, mu, sigma, n = params
    N_steps, dt, N_stat, nlags, N_thinning = hyperparams

    # Sample degree sequence
    degree_sequence = sample_lognormal(mu, sigma, N)
    K = int(np.mean(degree_sequence))
    
    # Build network
    A = adjacency_matrix(N, K, degree_sequence, device)
    J = inter_matrix(K, N, g, gamma, device)
    W = A * J
    
    # Initialize activity
    x = 2 * torch.rand(N, device=device) - 1
    traj = []
    
    for t in range(N_steps):
        x = step_dynamics(x, W, dt)
        if t >= N_steps - N_stat and t % N_thinning == 0:
            traj.append(x.detach().cpu())
    
    traj = torch.stack(traj)
    tau1 = ac_sim(traj, dt, nlags)
    
    return degree_sequence, tau1

# -----------------------------
# Parameters
# -----------------------------
N = 4000
g_list = [4]
sigma_list = [0.695]
mu_list = [3.83]
gamma_list = [0.35]
N_samples = 10

params = [
    (N, g, gamma, mu, sigma, n)
    for mu in mu_list
    for sigma in sigma_list
    for gamma in gamma_list
    for g in g_list
    for n in range(N_samples)
]

# Hyperparameters
N_steps = 10_000
dt = 0.1
N_stat = 5_000
nlags = 1_000
N_thinning = 1
hyperparams = (N_steps, dt, N_stat, nlags, N_thinning)

# -----------------------------
# Run simulations
# -----------------------------
degrees_model, tau_models = [], []

print("Running", len(params), "simulations...")

for p in params:
    deg, tau = simulate_lognormal_torch(p, hyperparams)
    degrees_model.append(deg)
    tau_models.append(tau)

degrees_model = np.concatenate(degrees_model)
tau_models = np.concatenate(tau_models)

# Save results
dm = {"degrees": degrees_model, "taus": tau_models}

#### Model autocorrelation functions

In [ ]:
import numpy as np
from scipy.integrate import odeint
from statsmodels.tsa.stattools import acf

# ---------------------------
# Activation functions
# ---------------------------
def phi(x):
    return np.tanh(x)

def dphi(x):
    return 1 - np.tanh(x)**2

# ---------------------------
# Dynamics
# ---------------------------
def dynamics(x, t, W):
    return -x + W @ phi(x)

# ---------------------------
# Lognormal sampling
# ---------------------------
def sample_lognormal(mu, sigma, size):
    return np.random.lognormal(mean=mu, sigma=sigma, size=size)

# ---------------------------
# Interaction matrix J
# ---------------------------
def inter_matrix(K, N, g, gamma):
    J = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            z = np.random.normal(size=2)
            J[i, j] = g/np.sqrt(K)*z[0]
            J[j, i] = g/np.sqrt(K)*(gamma*z[0]+np.sqrt(1-gamma**2)*z[1])
    return J

# ---------------------------
# Adjacency matrix A
# ---------------------------
def adjacency_matrix(N, K, degree_sequence):
    deg_seq = np.asarray(degree_sequence)
    prob_matrix = np.outer(deg_seq, deg_seq) / (K * N)
    upper_tri = np.triu_indices(N, k=1)
    mask = np.random.rand(len(upper_tri[0])) < prob_matrix[upper_tri]
    A = np.zeros((N, N), dtype=int)
    A[upper_tri[0][mask], upper_tri[1][mask]] = 1
    A += A.T
    return A

# ---------------------------
# Half-width at half-maximum
# ---------------------------
def HWHM(acf_values):
    acf_values = np.asarray(acf_values)
    if len(acf_values) < 2 or not np.isfinite(acf_values).all():
        return -1.0
    half = acf_values[0] / 2
    below_half = np.where(acf_values < half)[0]
    if len(below_half) == 0 or below_half[0] == 0:
        return -1.0
    i = below_half[0]
    y0, y1 = acf_values[i-1], acf_values[i]
    return i-1 + (half - y0) / (y1 - y0) if y1 != y0 else float(i-1)

# ---------------------------
# Compute autocorrelation HWHM
# ---------------------------
def ac_sim(sim, N_stat, dt, nlags):
    N = sim.shape[1]
    taus = np.zeros(N)
    for i in range(N):
        C = acf(sim[-N_stat:, i], nlags=nlags-1)
        taus[i] = HWHM(C) * dt
    return taus

# ---------------------------
# Main simulation
# ---------------------------
def simulate_lognormal(params, hyperparams):
    N, g, gamma, mu, sigma, n = params
    N_steps, dt, N_stat, nlags, N_thinning = hyperparams
    t = np.linspace(0, N_steps*dt, N_steps)

    degree_sequence = sample_lognormal(mu, sigma, N)
    K = int(np.mean(degree_sequence))

    A = adjacency_matrix(N, K, degree_sequence)
    J = inter_matrix(K, N, g, gamma)
    W = A * J

    ic = np.random.uniform(-1, 1, size=N)
    sim = odeint(dynamics, ic, t, args=(W,))
    
    tau1 = ac_sim(sim, N_stat, dt, nlags)
    return N, g, gamma, mu, sigma, n, tau1, degree_sequence

# ---------------------------
# Example parameters
# ---------------------------
N = 4000
g = 4.0
gamma = 0.35
mu = 3.83
sigma = 0.695
n = 1

params = (N, g, gamma, mu, sigma, n)
N_steps = 1500
dt = 0.1
N_stat = 1000
nlags = 500
N_thinning = 100
hyperparams = (N_steps, dt, N_stat, nlags, N_thinning)

# ---------------------------
# Run simulation
# ---------------------------
N, g, gamma, mu, sigma, n, acfs_model_curves, degree_sequence = simulate_lognormal(params, hyperparams)

#### ACFs comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------
def compute_binned_acfs(degree_sequence, acfs, n_bins=5):
    """Bin neurons by degree percentiles and compute mean ACF per bin."""
    bins = np.percentile(degree_sequence, np.linspace(0, 100, n_bins+1))
    bin_idx = np.digitize(degree_sequence, bins[1:-1])
    
    mean_acfs, labels = [], []
    for i in range(n_bins):
        idx = np.where(bin_idx == i)[0]
        if len(idx) == 0:
            mean_acfs.append(None)
            labels.append(None)
            continue
        mat = np.vstack([acfs[j] for j in idx])
        mean_acfs.append(mat.mean(axis=0))
        labels.append(f"{int(bins[i])}–{int(bins[i+1])}")
    return labels, mean_acfs

def plot_acf_with_inset(ax, mean_acfs, labels, palette, dt, inset_range=(0.45,0.55)):
    """Plot mean ACF curves with a zoom inset around a specific correlation range."""
    for i, acf in enumerate(mean_acfs):
        if acf is not None:
            ax.plot(np.arange(1, len(acf)) * dt, acf[1:] / acf[1], color=palette[i], label=labels[i])
    
    # Style main axis
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Lag [s]")
    ax.set_ylabel("ACF")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks([0, 5, 10])
    ax.set_yticks([0, 0.5, 1])
    
    # Inset
    axins = inset_axes(ax, width="40%", height="40%", loc='upper right')
    for i, acf in enumerate(mean_acfs):
        if acf is not None:
            axins.plot(np.arange(1, len(acf)) * dt, acf[1:] / acf[1], color=palette[i])
    
    # Determine inset limits based on mean across bins
    all_acfs = np.vstack([acf[1:] / acf[1] for acf in mean_acfs if acf is not None])
    mean_acf = all_acfs.mean(axis=0)
    idx_center = np.where((mean_acf > inset_range[0]) & (mean_acf < inset_range[1]))[0]
    if len(idx_center) > 0:
        x_min, x_max = idx_center[0]*dt, idx_center[-1]*dt
        axins.set_xlim(x_min, x_max)
        axins.set_ylim(*inset_range)
    
    # Style inset
    inset_ticks = np.linspace(x_min, x_max, 3)
    axins.set_xticks(inset_ticks)
    axins.set_xticklabels([f"{tick:.1f}" for tick in inset_ticks])
    axins.tick_params(axis='x', bottom=True, top=False, labelbottom=True)
    axins.spines['top'].set_visible(False)
    axins.spines['right'].set_visible(False)
    axins.set_yticks([0.5])
    
    return ax, axins

# ---------------------------------------------------------
# Compute binned ACFs
# ---------------------------------------------------------
labels_model, acfs_model_binned = compute_binned_acfs(degree_sequence, acfs_model)
labels_data,  acfs_data_binned  = compute_binned_acfs(df_all['in_degree'], df_all['acf'])

# Time steps
dt_model = 0.1*4/fps
dt_data  = 1/fps

# ---------------------------------------------------------
# Figure setup
# ---------------------------------------------------------
fig = plt.figure(figsize=(18/2.54, 5/2.54))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.3)

# Color palettes
palette_model = sns.color_palette("Oranges", n_colors=8)[3:]
palette_data  = sns.color_palette("Blues", n_colors=8)[3:]

# -------------------------
# Left: Model
# -------------------------
ax_model = fig.add_subplot(gs[0, 0])
ax_model, axins_model = plot_acf_with_inset(ax_model, acfs_model_binned, labels_model, palette_model, dt_model)
ax_model.text(-0.15, 1.05, "(a)", transform=ax_model.transAxes)
ax_model.text(0.5, 1.08, "Model", transform=ax_model.transAxes, ha='center', va='bottom', weight='bold')

# -------------------------
# Right: Data
# -------------------------
ax_data = fig.add_subplot(gs[0, 1])
ax_data, axins_data = plot_acf_with_inset(ax_data, acfs_data_binned, labels_data, palette_data, dt_data)
ax_data.text(-0.15, 1.05, "(b)", transform=ax_data.transAxes)
ax_data.text(0.5, 1.08, "Data", transform=ax_data.transAxes, ha='center', va='bottom', weight='bold')

# ---------------------------------------------------------
# Save figure
# ---------------------------------------------------------
plt.savefig('/PLACEHOLDER/panel_acf_insets_clean.png', dpi=1000, bbox_inches='tight')
plt.show()

### Loading and comparing the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# ---------------------------------------------------------
# Load and filter data
# ---------------------------------------------------------
df_matched = pd.read_pickle("/home/zenari/hwhm_in_degree_df_proof_1412.pkl")
fps = 6.3009

# Filter out neurons with too-short timescales
df_filtered = df_matched[df_matched['hwhm'] > (1 / fps)]

# Model data (assumes dm dictionary from previous simulation)
box_df_model = pd.DataFrame({'degrees': dm['degrees'], 'taus': dm['taus']})
box_df = df_filtered.copy()

# ---------------------------------------------------------
# Convert to percentile ranks
# ---------------------------------------------------------
for df, deg_col, tau_col in [(box_df, 'in_degree', 'hwhm'), (box_df_model, 'degrees', 'taus')]:
    df['deg_rank'] = df[deg_col].rank(pct=True)
    df['tau_rank'] = df[tau_col].rank(pct=True)

# ---------------------------------------------------------
# Bin degree ranks into n_bins
# ---------------------------------------------------------
n_bins = 5
edges = np.linspace(0, 1, n_bins + 1)
box_df['deg_bin'] = pd.cut(box_df['deg_rank'], edges, include_lowest=True)
box_df_model['deg_bin'] = pd.cut(box_df_model['deg_rank'], edges, include_lowest=True)

# ---------------------------------------------------------
# Prepare long-format dataframe for plotting
# ---------------------------------------------------------
df_plot = pd.concat([
    pd.DataFrame({'source':'model', 'deg_bin': box_df_model['deg_bin'], 'tau_val': box_df_model['tau_rank']}),
    pd.DataFrame({'source':'data',  'deg_bin': box_df['deg_bin'],       'tau_val': box_df['tau_rank']})
], ignore_index=True)

df_plot['deg_bin'] = df_plot['deg_bin'].cat.as_ordered()
order = df_plot['deg_bin'].cat.categories
hue_order = ['model', 'data']

# ---------------------------------------------------------
# Plot: Boxplots + Pointplots
# ---------------------------------------------------------
plt.figure(figsize=(8, 5), dpi=120)

ax = sns.boxplot(
    data=df_plot, x='deg_bin', y='tau_val', hue='source',
    order=order, hue_order=hue_order,
    showfliers=False, palette={'model':'lightblue','data':'steelblue'},
    zorder=1
)

sns.pointplot(
    data=df_plot, x='deg_bin', y='tau_val', hue='source',
    order=order, hue_order=hue_order,
    join=True, estimator='median', errorbar='se',
    palette={'model':'tab:green','data':'tab:red'},
    dodge=0.4, markers='o', linestyles='-',
    linewidth=1.6,
    zorder=5, ax=ax
)

# Single legend (remove duplicate)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title='', fontsize=13, frameon=False, loc=(0.35, 1), ncol=2)

# Axis labels
ax.set_xlabel("Degree rank (binned)", fontsize=13)
ax.set_ylabel("Timescale (percentile)", fontsize=13)
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# Spearman correlations
# ---------------------------------------------------------
rho_model, p_model = spearmanr(box_df_model['degrees'], box_df_model['taus'], nan_policy='omit')
rho_data, p_data     = spearmanr(box_df['in_degree'], box_df['hwhm'], nan_policy='omit')

print(f"Spearman model:  rho = {rho_model:.3f}, p = {p_model:.3g}")
print(f"Spearman data:   rho = {rho_data:.3f}, p = {p_data:.3g}")

### Loading data for hist plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Load data
# -----------------------------
df_degree = pd.read_pickle("/home/zenari/degree_df_proof_1412.pkl")
df_matched = pd.read_pickle("/home/zenari/hwhm_in_degree_df_proof_1412.pkl")

degree_data = df_degree['in_degree'].values
tau_data = df_matched['hwhm'].values
tau_data_kept = tau_data[tau_data > 1 / fps]

degree_model = dm['degrees']
tau_model    = dm['taus']

# -----------------------------
# Figure setup
# -----------------------------
fig = plt.figure(figsize=(18/2.54, 8/2.54), dpi=150)

# Axes positions: [left, bottom, width, height]
ax_a = fig.add_axes([0.09, 0.62, 0.18, 0.3])  # (a) Model degree histogram
ax_b = fig.add_axes([0.09, 0.16, 0.18, 0.3])  # (b) Model timescale histogram
ax_c = fig.add_axes([0.35, 0.2, 0.35, 0.7])   # (c) Central percentile plot
ax_d = fig.add_axes([0.8, 0.62, 0.18, 0.3])   # (d) Data degree histogram
ax_e = fig.add_axes([0.8, 0.16, 0.18, 0.3])   # (e) Data timescale histogram

# -----------------------------
# Colors
# -----------------------------
model_color_light = 'moccasin'
model_color_dark  = 'tab:orange'
data_color_light  = 'lightblue'
data_color_dark   = 'tab:blue'

palette_model = sns.color_palette("Oranges", n_colors=8)[3:]
palette_data  = sns.color_palette("Blues", n_colors=8)[3:]

# -----------------------------
# Panel (a) Model degree histogram
# -----------------------------
ax_a.hist(degree_model[degree_model < 300], bins=40, color=model_color_light, edgecolor='k', density=True)
ax_a.set_xlabel('Degree $k$')
ax_a.set_ylabel('$P(k)$')
ax_a.set_xlim(0, 300)
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)

# -----------------------------
# Panel (b) Model timescale histogram
# -----------------------------
tau_model_plot = tau_model * 2 / fps
tau_model_plot = tau_model_plot[tau_model_plot < 12]
ax_b.hist(tau_model_plot, bins=40, color=model_color_dark, edgecolor='k', density=True)
ax_b.set_xlabel('Timescale $\u03C4$ [s]')
ax_b.set_ylabel('$P(\\tau)$')
ax_b.set_xlim(2.1, 12)
ax_b.spines['top'].set_visible(False)
ax_b.spines['right'].set_visible(False)

# -----------------------------
# Panel (c) Model vs Data percentile plot
# -----------------------------
order = sorted(df_plot['deg_bin'].unique())
num_bins = len(order)
bin_edges = [0, 20, 40, 60, 80, 100]
percentile_labels = [f"{bin_edges[i]}-{bin_edges[i+1]}" for i in range(num_bins)]

# Compute median and IQR
stats = df_plot.groupby(['deg_bin', 'source'])['tau_val'].agg(
    ['median', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]
).reset_index()
stats.columns = ['deg_bin', 'source', 'median', 'q25', 'q75']

for src, palette, color in zip(['model', 'data'], [palette_model, palette_data], [model_color_dark, data_color_dark]):
    sub = stats[stats['source'] == src].sort_values('deg_bin')
    x_positions = np.arange(num_bins)
    x_positions = x_positions - 0.15 if src == 'model' else x_positions + 0.15

    # Plot IQR bars
    for i, (x, q25, q75) in enumerate(zip(x_positions, sub['q25'], sub['q75'])):
        ax_c.plot([x, x], [q25, q75], color=palette[i], linewidth=2, alpha=0.5)
        ax_c.plot([x - 0.06, x + 0.06], [q25, q25], color=palette[i], linewidth=1)
        ax_c.plot([x - 0.06, x + 0.06], [q75, q75], color=palette[i], linewidth=1)

    # Plot medians
    for i, (x, med) in enumerate(zip(x_positions, sub['median'])):
        ax_c.plot(x, med, marker='o', markersize=6, color=palette[i], linewidth=2, zorder=5)

    # Connect medians
    ax_c.plot(x_positions, sub['median'], color=color, linewidth=1, linestyle='-')

ax_c.set_xticks(np.arange(num_bins))
ax_c.set_xticklabels(percentile_labels)
ax_c.set_xlabel("Degree percentile")
ax_c.set_ylabel("Timescale [arbitrary units]")
ax_c.set_ylim(0, 1)

ax_c.legend(
    handles=[
        plt.Line2D([0], [0], color=model_color_dark, marker='o', linestyle='-'),
        plt.Line2D([0], [0], color=data_color_dark, marker='o', linestyle='-')
    ],
    labels=['Model', 'Data'],
    frameon=False,
    loc=(0.13, 1),
    ncol=2
)

# -----------------------------
# Panel (d) Data degree histogram
# -----------------------------
ax_d.hist(degree_data[degree_data < 300], bins=40, color=data_color_light, edgecolor='k', density=True)
ax_d.set_xlabel('In-degree')
ax_d.set_ylabel('$P(k)$')
ax_d.set_xlim(0, 300)
ax_d.set_yticks((0, 0.01))
ax_d.spines['top'].set_visible(False)
ax_d.spines['right'].set_visible(False)

# -----------------------------
# Panel (e) Data timescale histogram
# -----------------------------
tau_data_kept = tau_data[tau_data > 7/fps]
tau_data_kept = tau_data_kept[tau_data_kept < 12]
ax_e.hist(tau_data_kept, bins=30, color=data_color_dark, edgecolor='k', density=True)
ax_e.set_xlabel('Timescale [s]')
ax_e.set_ylabel('$P(\\tau)$')
ax_e.set_xlim(1, 12)
ax_e.spines['top'].set_visible(False)
ax_e.spines['right'].set_visible(False)

# -----------------------------
# Panel labels
# -----------------------------
ax_a.text(-0.3, 1, '(c)', transform=ax_a.transAxes, ha='left', va='bottom')
ax_b.text(-0.3, 1, '(d)', transform=ax_b.transAxes, ha='left', va='bottom')
ax_c.text(-0.2, 1.04, '(e)', transform=ax_c.transAxes, ha='left', va='bottom')
ax_d.text(-0.3, 1, '(f)', transform=ax_d.transAxes, ha='left', va='bottom')
ax_e.text(-0.3, 1, '(g)', transform=ax_e.transAxes, ha='left', va='bottom')

# Column titles
fig.text(0.17, 0.93, 'Model', ha='center', va='bottom', weight='bold')
fig.text(0.89, 0.93, 'Data', ha='center', va='bottom', weight='bold')

plt.tight_layout()
#plt.savefig('/home/zenari/RNN_heterogeneity/results/panel_microns_matched.jpg', dpi=1000)
plt.show()